# Faktor Lingkungan

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BASE = ROOT / "hasil_cleaning_final"
if not (BASE / "01_aqms_clean.csv").is_file():
    raise FileNotFoundError("Jalankan cleaning.ipynb terlebih dahulu.")
OUT = ROOT / "hasil_meteorologi_moniq"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
aqms = pd.read_csv(BASE / "01_aqms_clean.csv")
meteo = pd.read_csv(BASE / "02_meteorologi_clean.csv")
print("AQMS:", aqms.shape, "Meteorologi:", meteo.shape)
print(meteo.columns.tolist())


## Gabungkan data cuaca

In [ ]:
assert aqms.measurement_id.is_unique and meteo.measurement_id.is_unique
faktor = ["temperature_c", "humidity_pct", "wind_speed_ms", "wind_direction_deg"]
polutan = ["pm25_ug_m3", "pm10_ug_m3", "so2_ug_m3", "co_ug_m3", "o3_ug_m3", "no2_ug_m3"]
df = aqms.merge(meteo[["measurement_id"] + faktor], on="measurement_id", how="inner", validate="one_to_one")
df["tanggal"] = pd.to_datetime(df["tanggal"])
print("Baris tergabung:", len(df), "| Tidak cocok AQMS:", len(aqms)-len(df))


## Ringkasan data

In [ ]:
display(df[faktor + polutan].describe().T.round(2))
display(df[faktor + polutan].isna().mean().mul(100).round(2).rename("missing_pct"))


## Rata-rata harian

In [ ]:
keys = ["tanggal", "provinsi", "kota_kabupaten", "sensor_id"]
kolom = polutan + faktor[:3]
harian = df.groupby(keys, dropna=False)[kolom].mean().reset_index()
rad = np.deg2rad(df["wind_direction_deg"])
df["sin_arah"] = np.sin(rad)
df["cos_arah"] = np.cos(rad)
angin = df.groupby(keys, dropna=False)[["sin_arah", "cos_arah"]].mean().reset_index()
angin["wind_direction_deg"] = np.mod(np.rad2deg(np.arctan2(angin.sin_arah, angin.cos_arah)), 360)
harian = harian.merge(angin[keys + ["wind_direction_deg"]], on=keys, validate="one_to_one")
harian["bulan"] = harian.tanggal.dt.to_period("M").astype(str)
harian.to_csv(OUT / "01_meteorologi_polutan_harian_stasiun.csv", index=False)
print(harian.shape)
display(harian.head())


## Rata-rata bulanan

In [ ]:
bulanan = harian.groupby(["bulan", "provinsi", "kota_kabupaten"], dropna=False)[kolom].mean().reset_index()
bulanan.to_csv(OUT / "02_meteorologi_polutan_bulanan_wilayah.csv", index=False)
display(bulanan.head())


## Korelasi meteorologi

In [ ]:
hasil = []
for f in faktor[:3]:
    for p in polutan:
        t = harian[[f,p]].dropna()
        r, pv = pearsonr(t[f], t[p]) if len(t)>=3 and t[f].nunique()>1 and t[p].nunique()>1 else (np.nan,np.nan)
        hasil.append({"faktor":f,"polutan":p,"n_pasangan":len(t),"korelasi_pearson":r,"p_value":pv,"tingkat_analisis":"stasiun-hari"})
korelasi = pd.DataFrame(hasil)
korelasi.to_csv(OUT / "03_korelasi_meteorologi_polutan.csv", index=False)
display(korelasi.round(3))


## Arah angin

In [ ]:
arah = df[["wind_direction_deg", "wind_speed_ms", "pm25_ug_m3"]].dropna(subset=["wind_direction_deg"]).copy()
sektor = np.array(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
arah["arah_angin"] = sektor[np.floor(((arah.wind_direction_deg+22.5)%360)/45).astype(int)]
ringkasan_arah = arah.groupby("arah_angin").agg(jumlah_pengamatan=("wind_direction_deg","size"),rata_rata_kecepatan_angin_ms=("wind_speed_ms","mean"),rata_rata_pm25_ug_m3=("pm25_ug_m3","mean")).reindex(sektor).reset_index()
ringkasan_arah.to_csv(OUT / "04_ringkasan_arah_angin.csv",index=False)
display(ringkasan_arah)


In [ ]:
tren = bulanan.groupby("bulan")[["pm25_ug_m3", "temperature_c"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(tren.bulan, tren.pm25_ug_m3, label="PM2.5")
ax.set_ylabel("PM2.5 (µg/m³)")
ax2 = ax.twinx()
ax2.plot(tren.bulan, tren.temperature_c, "--", color="tab:orange", label="Suhu")
ax2.set_ylabel("Suhu (°C)")
step=max(1,len(tren)//8)
ax.set_xticks(range(0,len(tren),step)); ax.set_xticklabels(tren.bulan.iloc[::step],rotation=45)
plt.title("PM2.5 dan suhu bulanan — deskriptif")
plt.tight_layout()
plt.savefig(OUT / "grafik_tren_suhu_pm25.png", dpi=140)
plt.show()


In [ ]:
audit = pd.DataFrame([{"baris_aqms":len(aqms),"baris_meteorologi":len(meteo),"baris_tergabung":len(df),"baris_harian_stasiun":len(harian),"rentang_awal":str(df.tanggal.min().date()),"rentang_akhir":str(df.tanggal.max().date()),"catatan":"Korelasi eksploratif, tidak membuktikan kausalitas"}])
audit.to_csv(OUT / "05_audit_integrasi.csv",index=False)
display(audit)
